# OpenFF PEG Ether Fragments TorsionDrives v4.0

This notebook generates a TorsionDrive dataset of small poly(ethylene glycol) (PEG) ether fragments for a polymer fitting demo. The driven dihedral is the central `O-CH2-CH2-O` glycol torsion that governs PEG chain conformation.

The mapped SMILES and the driven `O-C-C-O` dihedral indices for the molecule set are frozen into the input file [`input.json`](input.json), which this notebook loads directly. `input.json` is produced by [`scripts/generate_fragments.py`](scripts/generate_fragments.py) (RDKit only).

`input.json` is a list of `{"mapped_smiles": ..., "dihedral_indices": [i, j, k, l]}` records (50 of them, one per molecule). The mapped SMILES carry atom map numbers = atom index + 1, so the 0-based `dihedral_indices` line up with the atom order recovered by `Molecule.from_mapped_smiles`.

In [1]:
import json
import pathlib

import numpy as np

from openff.toolkit import Molecule
from openff.qcsubmit import workflow_components
from openff.qcsubmit.factories import TorsiondriveDatasetFactory
from openff.qcsubmit.workflow_components import TorsionIndexer
from openff.qcsubmit.utils import get_symmetry_classes, get_symmetry_group

# Mapped SMILES + driven O-C-C-O dihedral indices for the demo fragment
# set, frozen into input.json (produced by scripts/generate_fragments.py).
INPUT_FILE = pathlib.Path("input.json")

## Molecule preparation

`input.json` holds 50 records, one per molecule. For each record:

- `mapped_smiles` is the canonical isomeric explicit-hydrogen mapped SMILES (atom map number = index + 1).
- `dihedral_indices` are the 0-based `(O, C, C, O)` indices of the central glycol torsion to drive, into that same atom order.
- `Molecule.from_mapped_smiles(mapped_smiles, allow_undefined_stereo=True)` rebuilds the OpenFF molecule with the **same** atom order, so the indices line up.

Before attaching the torsion we verify the 4 atoms are bonded `O-C-C-O`.

Stereochemistry: not enumerated (the fragments are achiral / racemic by construction; `allow_undefined_stereo=True`). Tautomers: not enumerated. Conformers: generated below with `StandardConformerGenerator(max_conformers=5)`.

In [2]:
def load_molecules():
    """Load the PEG ether fragment library from input.json and attach the
    central O-C-C-O torsion to each."""
    records = json.loads(INPUT_FILE.read_text())

    molecules = []
    for record in records:
        mapped = record["mapped_smiles"]
        i, j, k, l = record["dihedral_indices"]

        mol = Molecule.from_mapped_smiles(mapped, allow_undefined_stereo=True)

        # Verify the 4 atoms are bonded O-C-C-O before adding.
        bonded = all(
            mol.get_bond_between(a, b) is not None
            for a, b in ((i, j), (j, k), (k, l))
        )
        symbols = [mol.atoms[a].symbol for a in (i, j, k, l)]
        if not bonded or symbols != ["O", "C", "C", "O"]:
            print(f"  skipping {mapped}: not a bonded O-C-C-O ({symbols})")
            continue

        symmetry_classes = get_symmetry_classes(mol)
        indexer = TorsionIndexer()
        central_bond = (j, k)  # middle two atoms of the O-C-C-O
        sym_group = get_symmetry_group(central_bond, symmetry_classes)
        indexer.add_torsion(
            torsion=(i, j, k, l),
            symmetry_group=sym_group,
            scan_range=(-165, 180),
            scan_increment=[15],
        )
        mol.properties["dihedrals"] = indexer
        molecules.append(mol)

    return molecules


molecules = load_molecules()
print(f"Prepared {len(molecules)} molecules with attached O-C-C-O torsions.")

Prepared 50 molecules with attached O-C-C-O torsions.


## Dataset generation

Conformers are generated with `StandardConformerGenerator(max_conformers=5)`.

In [3]:
factory = TorsiondriveDatasetFactory()
factory.add_workflow_components(
    workflow_components.StandardConformerGenerator(max_conformers=5)
)

# Molecular weight stats for the long description (STANDARDS.md requires
# min/mean/max MW in the QCArchive metadata). Computed from the frozen
# molecule set; no molecules are filtered, so these match dataset.molecules.
masses = np.array([sum(atom.mass.m for atom in mol.atoms) for mol in molecules])

description = (
    "A torsiondrive dataset of small poly(ethylene glycol) (PEG) ether "
    "fragments generated to provide QM reference data for the central "
    "O-CH2-CH2-O glycol torsion that governs PEG chain conformation. Each "
    "molecule is a fragment of a long PEG chain, T-(O-CH2-CH2)_n-O-T', "
    "sampled across variable length (n = 1-4), end-group registration "
    "(hydroxyl or an sp3 carbon terminus, chosen independently on each "
    "end) and an occasional methyl branch. This yields the varied COCCO / "
    "COCCOC / CCOCCOC / OCCOCCOC family while always retaining a clean "
    "O-CH2-CH2-O torsion to drive. This dataset was generated for a "
    "polymer fitting demo.\n\n"
    "This dataset was generated at the B3LYP-D3BJ/DZVP level of theory "
    "using Psi4 with no implicit solvent and default specs. It includes "
    "molecules containing the elements C, H, and O, all with a formal "
    "charge of 0. "
    f"The molecular weights range from {masses.min():.2f} amu to "
    f"{masses.max():.2f} amu, with a mean of {masses.mean():.2f} amu."
)

dataset = factory.create_dataset(
    dataset_name="OpenFF PEG Ether Fragments TorsionDrives v4.0",
    tagline="Torsion drives of the O-CH2-CH2-O glycol torsion in small PEG ether fragments.",
    description=description,
    molecules=molecules,
)

dataset.metadata.submitter = "lilyminium"
dataset.metadata.long_description_url = (
    "https://github.com/openforcefield/qca-dataset-submission/tree/master/"
    "submissions/" + pathlib.Path.cwd().name
)

Deduplication                 :   0%|                    | 0/50 [00:00<?, ?it/s]

Deduplication                 : 100%|█████████| 50/50 [00:00<00:00, 2659.98it/s]

StandardConformerGenerator    :   0%|                    | 0/50 [00:00<?, ?it/s]

StandardConformerGenerator    :   2%|▏           | 1/50 [00:06<05:37,  6.88s/it]

StandardConformerGenerator    :  48%|█████▎     | 24/50 [00:06<00:05,  4.81it/s]

...

Preparation                   :  62%|██████▊    | 31/50 [00:00<00:00, 68.69it/s]

Preparation                   :  78%|████████▌  | 39/50 [00:00<00:00, 48.28it/s]

Preparation                   :  90%|█████████▉ | 45/50 [00:00<00:00, 43.19it/s]

Preparation                   : 100%|███████████| 50/50 [00:00<00:00, 42.08it/s]

Preparation                   : 100%|███████████| 50/50 [00:00<00:00, 50.53it/s]

In [4]:
dataset.add_qc_spec(
    method="B3LYP-D3BJ",
    basis="DZVP",
    program="psi4",
    spec_name="default",
    spec_description="Standard OpenFF optimization quantum chemistry specification.",
    store_wavefunction="none",
    implicit_solvent=None,
    maxiter=200,
    scf_properties=[
        "dipole",
        "quadrupole",
        "wiberg_lowdin_indices",
        "mayer_indices",
        "lowdin_charges",
        "mbis_charges",
    ],
    keywords={},
    overwrite=True,
)

## Validation stats (for the README)

In [5]:
confs = np.array([len(mol.conformers) for mol in dataset.molecules])
print("* Number of unique molecules:", dataset.n_molecules)
print("* Number of driven torsions:", dataset.n_records)
print("* Number of filtered molecules:", dataset.n_filtered)
print("* Number of conformers:", int(confs.sum()))
print(
    "* Number of conformers per molecule (min, mean, max): "
    f"{confs.min()}, {confs.mean():.2f}, {confs.max()}"
)

masses = np.array(
    [sum(atom.mass.m for atom in molecule.atoms) for molecule in dataset.molecules]
)
print(f"* Mean molecular weight: {masses.mean():.2f}")
print(f"* Min molecular weight: {masses.min():.2f}")
print(f"* Max molecular weight: {masses.max():.2f}")
print("* Charges:", sorted(set(m.total_charge.m for m in dataset.molecules)))
print(f"* Elements: {{{', '.join(dataset.metadata.dict()['elements'])}}}")

fields = ["basis", "implicit_solvent", "keywords", "maxiter", "method", "program"]
for spec, obj in dataset.qc_specifications.items():
    od = obj.dict()
    print("* Spec:", spec)
    for field in fields:
        print(f"\t * {field}: {od[field]}")
    print(f"\t * driver: {factory.driver.name}")
    print("\t* SCF properties:")
    for field in od["scf_properties"]:
        print(f"\t\t* {field}")

* Number of unique molecules: 50
* Number of driven torsions: 50
* Number of filtered molecules: 0
* Number of conformers: 129
* Number of conformers per molecule (min, mean, max): 1, 2.58, 5
* Mean molecular weight: 146.89
* Min molecular weight: 62.07
* Max molecular weight: 204.31
* Charges: [0.0]
* Elements: {O, C, H}
* Spec: default
	 * basis: DZVP
	 * implicit_solvent: None
	 * keywords: {}
	 * maxiter: 200
	 * method: B3LYP-D3BJ
	 * program: psi4
	 * driver: gradient
	* SCF properties:
		* dipole
		* quadrupole
		* wiberg_lowdin_indices
		* mayer_indices
		* lowdin_charges
		* mbis_charges


## Export

Writes `dataset.json.bz2` (the compressed submission), `dataset.smi` (one SMILES per molecule) and `dataset.pdf` (2D structures with the driven torsion highlighted). The PDF step is wrapped so that a missing cairo / toolkit backend does not prevent the JSON + SMI outputs from being produced.

In [6]:
dataset.export_dataset("dataset.json.bz2")
dataset.molecules_to_file("dataset.smi", "smi")
try:
    dataset.visualize("dataset.pdf", columns=8)
    print("Wrote dataset.pdf")
except Exception as exc:
    print(f"WARNING: dataset.visualize (PDF) failed and was skipped: {exc!r}")

Wrote dataset.pdf
